# Archive historique — proposer_annotations

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Propose des annotations depuis les textes PDF, indépendamment des CSV existants.

Les propositions restent à vérifier humainement avant de devenir une référence.
Script historique : les règles actives sont intégrées à `../pipeline_extraction_nominations.py`.
"""

from __future__ import annotations

import csv
import re
import unicodedata
from bisect import bisect_right
from pathlib import Path


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"
TITRE = re.compile(
    r"(?im)^[ \t\f]*(?:II|III|IV)(?:\.\d+)?\.[ \t]*NOMINATION[S]?\b"
)
SOUS_TITRE = re.compile(
    r"(?im)^[ \t\f]*(?:II|III|IV)(?:\.\d+)?\.[ \t]*(?:NOMINATION[S]?|REVOCATIONS|FIN DE FONCTIONS)\b[^\n]*"
)
PERSONNE = re.compile(
    r"\b(Monsieur|Monsioeur|Madame)\s+"
    r"((?:(?!\bMonsieur\b|\bMonsioeur\b|\bMadame\b).){2,120}?)"
    r"(?=,\s*|;\s*|\s+Mle\b|\.\s+(?:Le\s+|Au\s+|Page\s+|ADMINISTRATEUR|[A-Z]\.\s+AU|$))",
    re.IGNORECASE | re.DOTALL,
)
MATRICULE = re.compile(r"^\s*,?\s*M(?:le)?\s+([A-Z0-9][A-Z0-9\s\-]{1,28}?)\s*[,;(]")
VERBE = re.compile(r"\best\s+nomm(?:é|ée)\s+", re.IGNORECASE)
GROUPE = re.compile(
    r"(?i)(?:sont\s+nomm(?:és|ées)|sont\s+désign(?:és|ées))\s+(.{1,500}?)(?:\s*:\s*|\.\s+)"
    , re.DOTALL
)
BULLET = re.compile(r"(?m)^\s*[-–]\s*(?=Monsieur|Monsioeur|Madame)", re.IGNORECASE)
SIGNATURE = re.compile(r"(?im)^\s*Le Ministre de la Communication et\s*$")
MOIS = {
    "janvier": "01", "fevrier": "02", "mars": "03", "avril": "04",
    "mai": "05", "juin": "06", "juillet": "07", "aout": "08",
    "septembre": "09", "octobre": "10", "novembre": "11", "decembre": "12",
}


def normaliser_espace(texte: str) -> str:
    return re.sub(r"\s+", " ", texte).strip()


def date_seance(texte: str, nom_fichier: str) -> tuple[str, str]:
    debut = "".join(c for c in unicodedata.normalize("NFKD", texte[:4500].lower())
                    if not unicodedata.combining(c))
    motif = re.compile(
        r"\ble\s+(\d{1,2})(?:er)?\s+"
        r"(janvier|fevrier|mars|avril|mai|juin|juillet|aout|septembre|octobre|novembre|decembre)"
        r"\s+(20\d{2})\b"
    )
    match = motif.search(debut)
    if not match:
        return nom_fichier, "date_tiree_du_nom_de_fichier_a_verifier"
    date = f"{match[3]}-{MOIS[match[2]]}-{int(match[1]):02d}"
    return date, "date_du_document_differe_du_nom_de_fichier" if date != nom_fichier else ""


def lire_csv(chemin: Path) -> list[dict[str, str]]:
    with chemin.open(newline="", encoding="utf-8-sig") as fichier:
        return list(csv.DictReader(fichier))


def dans_revocation(texte: str, position: int) -> bool:
    titres = list(SOUS_TITRE.finditer(texte, 0, position))
    return bool(titres and re.search(r"REVOCATION|FIN DE FONCTIONS", titres[-1].group(0), re.IGNORECASE))


def type_nomination(texte: str, position: int) -> str:
    titres = list(SOUS_TITRE.finditer(texte, 0, position))
    if titres and "CONSEILS D" in titres[-1].group(0).upper():
        return "conseil_administration"
    return "fonction_individuelle"


def extrait(texte: str, debut: int, fin_nom: int) -> str:
    suite = texte[fin_nom:min(len(texte), fin_nom + 700)]
    fin = re.search(r";|\.\s*(?=\n|$)", suite)
    fin_position = fin_nom + fin.end() if fin else min(len(texte), fin_nom + 350)
    return texte[debut:fin_position].strip()


def fonction_directe(annonce: str) -> str:
    annonce = re.sub(r"(?i)Page\s+\d+\s+sur\s+\d+", " ", annonce).replace("\f", " ")
    match = VERBE.search(annonce)
    if not match:
        return ""
    reste = annonce[match.end():]
    reste = re.split(r"(?i)\s+en\s+remplacement\s+de\s+|\s*;\s*|\.\s*$", reste, maxsplit=1)[0]
    return normaliser_espace(reste)


def fonction_groupe(texte: str, debut: int) -> str:
    amont = texte[:debut]
    groupes = list(GROUPE.finditer(amont))
    if not groupes:
        return ""
    proposition = normaliser_espace(groupes[-1].group(1))
    # Les listes commencent souvent par « au poste de ... » ou se terminent
    # par « les personnes dont les noms suivent ».
    proposition = re.sub(r"(?i)^au poste de\s+", "", proposition)
    proposition = re.sub(r"(?i),?\s*les personnes dont les noms suivent\s*$", "", proposition)
    proposition = re.sub(r"(?i),?\s*les [^,]{0,100} dont les noms suivent\s*$", "", proposition)
    proposition = re.split(r"(?i)\s+pour\s+(?:un|une)\s+(?:premier|premi[èe]re|dernier|derni[èe]re)\s+mandat", proposition)[0]
    return proposition


def fonction_conseil(texte: str, debut: int, apres_nom: str, est_puce: bool) -> str:
    # Dans les décrets individuels, le rôle suit souvent les qualificatifs.
    role_direct = re.search(
        r"(?is)\bAdministrat(?:eur|rice)\b.{0,450}?"
        r"(?=\s+pour\s+un(?:e)?\s+|\s+en\s+remplacement\s+de\s+|[;.])",
        apres_nom,
    )
    if role_direct and not est_puce:
        return normaliser_espace(re.sub(r"(?i)Page\s+\d+\s+sur\s+\d+", " ", role_direct.group(0)))
    roles = list(re.finditer(
        r"(?i)\b(?:Administrateurs?\s+(?:repr[ée]sentant|au\s+Conseil)|"
        r"Administratrice\s+(?:repr[ée]sentant|au\s+Conseil)|"
        r"Pr[ée]sident(?:e)?\s+du\s+Conseil)\b"
        r".{0,180}?(?=\s+pour\s+un\s+|\s+pour\s+une\s+|[;.])",
        apres_nom,
        re.DOTALL,
    ))
    if roles and not est_puce:
        role = normaliser_espace(re.sub(r"(?i)Page\s+\d+\s+sur\s+\d+", " ", roles[-1].group(0)))
        if role:
            return role

    # Les listes portent parfois un intitulé collectif au-dessus des noms.
    amont = texte[:debut].replace("\f", "\n")
    titres = list(re.finditer(
        r"(?im)^[ \t]*ADMINISTRATEURS?\s+REPR[ÉE]SENTANT\b",
        amont,
    ))
    decrets = list(re.finditer(
        r"(?is)\b(?:nomme|nomination\s+des?|renouvelle\s+le\s+mandat\s+des?)\b"
        r".{0,220}?\b(Administrateurs?\b.{0,150}?)(?=\s+au\s+Conseil|\s+pour\s+un|\s+pour\s+une|[;.])",
        amont,
    ))
    dernier_titre = titres[-1] if titres else None
    dernier_decret = decrets[-1] if decrets else None
    if dernier_decret and (not dernier_titre or dernier_decret.start() > dernier_titre.start()):
        return normaliser_espace(decrets[-1].group(1))
    if dernier_titre:
        suffixe = amont[dernier_titre.start():min(debut, dernier_titre.start() + 250)]
        fin = re.search(r"(?im)(?:\n\s*[-–]\s*(?:Monsieur|Madame)|\n\s*Au\s+titre|\n\s*Le\s+Conseil|\f)", suffixe)
        role = normaliser_espace(suffixe[:fin.start()] if fin else suffixe)
        if role and len(role) < 170:
            return role
    # Pour les listes réparties sur plusieurs pages, reprendre le rôle annoncé
    # par le dernier décret, avant la liste des personnes.
    decrets_listes = list(re.finditer(
        r"(?i)\bLe\s+(?:premier|second|deuxième|troisième|quatrième|cinquième|"
        r"sixième|septième|huitième|neuvième|dixième)\s+décret\b|"
        r"\bLe\s+Conseil\s+a\s+adopté\s+un\s+décret\b",
        amont,
    ))
    if est_puce and decrets_listes:
        annonce = amont[decrets_listes[-1].start():]
        role_liste = re.search(
            r"(?is)\b(Administrateurs?\s+.{0,500}?\bau\s+Conseil\s+"
            r"d[’']administration\s+d[eu]\s+.{0,220}?)"
            r"(?=\s+pour\s+un(?:e)?\s+|[;.])",
            annonce,
        )
        if role_liste:
            return normaliser_espace(role_liste.group(1))
    return ""


def ministere_amont(texte: str, debut: int) -> str:
    amont = texte[:debut].replace("\f", "\n")
    titres = list(re.finditer(
        r"(?m)^[ \t]*(?:[A-Z]\.\s*)?"
        r"(?:AU\s+TITRE\s+(?:(?:DU|DE LA|DES|DE L[’'])\s+)?)?"
        r"(?:MINISTERE|PRIMATURE|PRESIDENCE|PREMIER\s+MINISTERE)\b",
        amont,
    ))
    if not titres:
        return ""
    depart = titres[-1].start()
    suite = amont[depart:min(debut, depart + 500)]
    lignes = []
    for ligne in suite.splitlines():
        ligne = ligne.strip()
        if not ligne:
            continue
        if any(caractere.islower() for caractere in ligne) or re.match(r"(?i)Page\s+\d", ligne):
            break
        lignes.append(ligne)
        if len(lignes) >= 4:
            break
    valeur = normaliser_espace(" ".join(lignes))
    valeur = re.sub(r"^(?:[A-Z]\.\s*)?AU TITRE (?:(?:DU|DE LA|DES|DE L[’'])\s+)?", "", valeur)
    valeur = re.sub(r"^[A-Z]\.\s*", "", valeur)
    return valeur if len(valeur) <= 180 else ""


def proposer_document(document: dict[str, str]) -> list[dict[str, str]]:
    nom_document = document["document_id"]
    texte = (RACINE / document["texte_pour_annotation"]).read_text(encoding="utf-8")
    return proposer_texte(texte, nom_document, document["date_nom_fichier"])


def proposer_texte(texte: str, nom_document: str, date_nom_fichier: str) -> list[dict[str, str]]:
    """Propose des nominations depuis un texte déjà extrait d'un PDF.

    Cette fonction ne lit ni n'écrit de fichier. Elle permet à la pipeline
    d'appliquer les mêmes règles à un PDF isolé ou à un répertoire complet.
    """
    titre = TITRE.search(texte)
    if not titre:
        return []
    debut_section = titre.start()
    signatures = list(SIGNATURE.finditer(texte, debut_section))
    fin_section = signatures[-1].start() if signatures else len(texte)
    section = texte[debut_section:fin_section]
    date, note_date = date_seance(texte, date_nom_fichier)
    pages = [i for i, caractere in enumerate(texte) if caractere == "\f"]
    puces = list(BULLET.finditer(section))
    lignes = []
    for personne in PERSONNE.finditer(section):
        if dans_revocation(section, personne.start()):
            continue
        avant_nom = section[max(0, personne.start() - 250):personne.start()]
        avant_nom = re.sub(r"(?i)Page\s+\d+\s+sur\s+\d+", " ", avant_nom)
        avant_nom = normaliser_espace(avant_nom)
        if re.search(r"(?i)en\s+remplacement\s+de\s*$", avant_nom):
            continue
        civilite, nom_brut = personne.group(1), personne.group(2)
        civilite = "Monsieur" if civilite.lower() == "monsioeur" else civilite
        nom = normaliser_espace(nom_brut)
        fin_personne = personne.end()
        # Certaines extractions PDF placent une virgule dans un nom composé,
        # par exemple « Monsieur Aboubakar, TRAORE, Mle ... ». Rattacher la
        # partie nominale avant de chercher le matricule.
        suite_nom = section[fin_personne:fin_personne + 100]
        fragment_nom = re.match(
            r"\s*,\s*([A-ZÀÂÇÉÈÊËÎÏÔÖÙÛÜŸ][A-ZÀÂÇÉÈÊËÎÏÔÖÙÛÜŸ'’\-\s]{1,80}?)\s*,\s*Mle\b",
            suite_nom,
        )
        if fragment_nom:
            nom = normaliser_espace(nom + " " + fragment_nom.group(1))
            fin_personne += fragment_nom.end(1)
        if (
            len(nom) < 3 or len(nom) > 100 or "Page " in nom
            or re.search(r"(?i)\b(?:Le Conseil|Minist[eè]re|D[eé]cret|nomm[ée]|dont\s+[a-z]+\s+ach[eè]ve)\b", nom)
        ):
            continue
        puce = next((p for p in reversed(puces) if p.end() == personne.start()), None)
        est_puce = puce is not None
        debut = puce.start() if puce else max(0, personne.start() - 90)
        passage = extrait(section, debut, fin_personne)
        suite = section[fin_personne:fin_personne + 70]
        matricule = MATRICULE.match(suite)
        matricule = normaliser_espace(matricule.group(1)) if matricule else ""
        fonction = fonction_directe(passage)
        origine_fonction = "directe" if fonction else ""
        typ = type_nomination(section, personne.start())
        if not fonction and est_puce and typ == "fonction_individuelle":
            fonction = fonction_groupe(section, debut)
            origine_fonction = "groupe" if fonction else ""
        if not fonction and typ == "conseil_administration":
            # Ne jamais emprunter le rôle de la personne annoncée ensuite.
            fin_passage = debut + len(passage)
            apres_nom = section[fin_personne:fin_passage]
            fonction = fonction_conseil(section, debut, apres_nom, est_puce)
            origine_fonction = "conseil_contexte" if fonction else ""
        # Les listes de préfets indiquent le département sur chaque ligne.
        if re.fullmatch(r"(?i)Préfet(?: de [Dd]épartement)?", fonction):
            lieu = re.search(r"(?i)\bD[ée]partement\s+de\s+([^;.]+)", passage)
            if lieu:
                fonction = "Préfet du département de " + normaliser_espace(lieu.group(1))
        abs_pos = debut_section + personne.start()
        page = bisect_right(pages, abs_pos) + 1
        ministere = ministere_amont(section, debut)
        notes = []
        if not est_puce:
            notes.append("nom_hors_liste_a_verifier")
        if not fonction:
            notes.append("fonction_a_completer")
        elif origine_fonction != "directe":
            notes.append("fonction_deduite_du_contexte_a_verifier")
        if not ministere:
            notes.append("ministere_a_completer_ou_non_indique")
        if note_date:
            notes.append(note_date)
        lignes.append({
            "document_id": nom_document,
            "page": str(page),
            "type_nomination": typ,
            "date_conseil": date,
            "civilite": civilite.title(),
            "nom": nom,
            "matricule": matricule,
            "fonction": fonction,
            "ministere": ministere,
            "extrait_exact": passage,
            "commentaire": "; ".join(notes),
            "origine_fonction": origine_fonction,
            "mode_proposition": "puce" if est_puce else "texte",
            "statut_verification": "a_verifier",
        })
    return lignes


def main() -> None:
    documents = lire_csv(DOSSIER / "documents_echantillon.csv")
    propositions = []
    for document in documents:
        if document["statut_annotation"] == "a_faire":
            propositions.extend(proposer_document(document))
    sortie = DOSSIER / "annotations_proposees.csv"
    colonnes = [
        "document_id", "page", "type_nomination", "date_conseil", "civilite",
        "nom", "matricule", "fonction", "ministere", "extrait_exact",
        "commentaire", "origine_fonction", "mode_proposition", "statut_verification",
    ]
    with sortie.open("w", newline="", encoding="utf-8-sig") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=colonnes)
        writer.writeheader()
        writer.writerows(propositions)
    print(f"{len(propositions)} propositions dans {sortie}")


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()